In [6]:
# 노트북 실행에 필요한 패키지 설치
%pip install -q python-dotenv openai langchain-core langchain-classic langchain-openai langchain-teddynote

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 날짜 형식 출력 파서

In [7]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_classic.output_parsers import DatetimeOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("CH03-OutputParser")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH03-OutputParser


In [9]:
output_parser = DatetimeOutputParser()
output_parser.format = "%Y-%m-%d"

In [10]:
print(output_parser.get_format_instructions())

Write a datetime string that matches the following pattern: '%Y-%m-%d'.

Examples: 2026-09-16, 2025-09-16, 2026-09-15

Return ONLY this string, no other words!


In [12]:
template = """Answer the users question:

#Format Instructions:
{format_instructions}

#Question:
{question}

#Answer:"""

prompt = PromptTemplate.from_template(
    template,
    partial_variables={
        "format_instructions": output_parser.get_format_instructions()
    }, # 지침을 템플릿에 적용
)

prompt # 프롬프트 내용을 출력

PromptTemplate(input_variables=['question'], input_types={}, partial_variables={'format_instructions': "Write a datetime string that matches the following pattern: '%Y-%m-%d'.\n\nExamples: 2026-09-16, 2025-09-16, 2026-09-15\n\nReturn ONLY this string, no other words!"}, template='Answer the users question:\n\n#Format Instructions:\n{format_instructions}\n\n#Question:\n{question}\n\n#Answer:')

In [13]:
chain = prompt | ChatOpenAI() | output_parser # 프롬프트와 LLM, 출력 파서를 연결

output = chain.invoke({"question": "구글이 창업한 연도?"}) # 체인을 실행

In [15]:
output.strftime("%Y-%m-%d") # 결과를 문자열로 반환

'1998-09-04'

### 열거형 출력 파서

In [19]:
from enum import Enum
from langchain_classic.output_parsers import EnumOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("CH03-OutputParser")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH03-OutputParser


In [20]:
class Colors(Enum):
    RED = "빨간색"
    GREEN = "초록색"
    BLUE = "파란색"

In [21]:
Colors.RED # 빨간색 항목에 접근하기

<Colors.RED: '빨간색'>

In [23]:
parser = EnumOutputParser(enum=Colors) # EnumOutputParser를 생성하고 Colors Enum을 전달
parser.get_format_instructions() # EnumOutputParser의 지침을 확인

'Select one of the following options: 빨간색, 초록색, 파란색'

In [24]:
prompt = PromptTemplate.from_template(
    """다음의 물체는 어떤 색깔인가요?

Object: {object}

Instructions: {instructions}"""
).partial(instructions=parser.get_format_instructions()) # EnumOutputParser의 지침을 프롬프트에 적용

chain = prompt | ChatOpenAI() | parser # 프롬프트와 LLM, 출력 파서를 연결

In [25]:
response = chain.invoke({"object": "하늘"}) # 체인을 실행
print(response) # 결과를 출력

Colors.BLUE


In [26]:
type(response) # 결과의 타입을 확인

<enum 'Colors'>

In [27]:
response.value # Enum 항목의 값을 확인

'파란색'